# Phase 2: Preprocessing & Feature Engineering
**Objective:** Clean the dataset, group clinical features, prevent data leakage, and prepare the matrices for model training. This notebook outputs the final `X_train`, `X_val`, `X_test` (and their `y` counterparts) to `data/processed/`.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set plotting style for professional visualizations
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.figsize': (10, 6), 'axes.titlesize': 14})

print("Libraries imported successfully.")

Libraries imported successfully.


In [7]:
# Load the raw dataset downloaded via Phase 0 ingestion script
# Note: The UCI Diabetes dataset uses '?' to denote missing values. 
# We will replace them with np.nan immediately upon loading for pandas compatibility.

file_path = '../data/raw/diabetes_130_us_raw.csv'
df = pd.read_csv(file_path, low_memory=False, na_values=['?'])

print(f"Dataset loaded. Shape: {df.shape}")
display(df.head())

Dataset loaded. Shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


# Phase 2: Preprocessing & Feature Engineering
**Objective:** Clean the dataset, group clinical features, prevent data leakage, and prepare the matrices for model training. This notebook outputs the final `X_train`, `X_val`, `X_test` (and their `y` counterparts) to `data/processed/`.

In [8]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from imblearn.over_sampling import SMOTE

# Load the raw data
file_path = '../data/raw/diabetes_130_us_raw.csv'
df = pd.read_csv(file_path, low_memory=False, na_values=['?'])

print(f"Initial Shape: {df.shape}")

Initial Shape: (101766, 50)


In [9]:
# 1. Target Variable: Binary Classification (1 = '<30', 0 = '>30' or 'NO')
df['readmitted'] = df['readmitted'].apply(lambda x: 1 if x == '<30' else 0)

# 2. Prevent Data Leakage: Keep only the first encounter for each patient
df = df.sort_values('encounter_id').drop_duplicates(subset=['patient_nbr'], keep='first')

print(f"Shape after deduplication: {df.shape}")
print("Target Distribution:\n", df['readmitted'].value_counts(normalize=True))

Shape after deduplication: (71518, 50)
Target Distribution:
 readmitted
0    0.912008
1    0.087992
Name: proportion, dtype: float64


In [10]:
# 1. Drop 'weight' due to >96% missingness
if 'weight' in df.columns:
    df = df.drop(columns=['weight'])

# 2. Fill missing categorical values with 'Unknown'
cols_to_fill = ['medical_specialty', 'payer_code', 'race']
for col in cols_to_fill:
    df[col] = df[col].fillna('Unknown')

# 3. Drop rows where essential clinical identifiers are missing (e.g., gender='Unknown/Invalid')
df = df[df['gender'] != 'Unknown/Invalid']

In [11]:
# The original dataset uses highly granular ICD-9 codes. 
# We group them into 9 primary categories based on the Strack et al. methodology.

def map_icd9(val):
    if pd.isna(val): return 'Missing'
    
    # Handle string values that start with 'V' or 'E' (Supplementary classifications)
    if str(val).startswith('V') or str(val).startswith('E'): return 'Other'
    
    try:
        # Convert to float for numerical range checking
        v = float(val)
        if 390 <= v <= 459 or v == 785: return 'Circulatory'
        if 460 <= v <= 519 or v == 786: return 'Respiratory'
        if 520 <= v <= 579 or v == 787: return 'Digestive'
        if np.floor(v) == 250: return 'Diabetes'
        if 800 <= v <= 999: return 'Injury'
        if 710 <= v <= 739: return 'Musculoskeletal'
        if 580 <= v <= 629 or v == 788: return 'Genitourinary'
        if 140 <= v <= 239: return 'Neoplasms'
    except ValueError:
        pass
    
    return 'Other'

# Apply mapping to primary, secondary, and tertiary diagnoses
for col in ['diag_1', 'diag_2', 'diag_3']:
    df[f'{col}_group'] = df[col].apply(map_icd9)

# Drop the original granular columns and IDs
df = df.drop(columns=['diag_1', 'diag_2', 'diag_3', 'encounter_id', 'patient_nbr'])

In [12]:
# Extract target
y = df['readmitted']
X = df.drop(columns=['readmitted'])

# Split 1: Train (70%) and Temp (30%)
# Stratify ensures the severe class imbalance is maintained across all splits
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# Split 2: Validation (15%) and Test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"X_test shape:  {X_test.shape}")

X_train shape: (50060, 46)
X_val shape:   (10727, 46)
X_test shape:  (10728, 46)


In [16]:
# Identify numerical and categorical columns
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

# 1. Scale Numerical Features
scaler = StandardScaler()
X_train_num = pd.DataFrame(scaler.fit_transform(X_train[num_cols]), columns=num_cols, index=X_train.index)
X_val_num = pd.DataFrame(scaler.transform(X_val[num_cols]), columns=num_cols, index=X_val.index)
X_test_num = pd.DataFrame(scaler.transform(X_test[num_cols]), columns=num_cols, index=X_test.index)

# 2. Encode Categorical Features
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_cat = pd.DataFrame(encoder.fit_transform(X_train[cat_cols]), columns=encoder.get_feature_names_out(cat_cols), index=X_train.index)
X_val_cat = pd.DataFrame(encoder.transform(X_val[cat_cols]), columns=encoder.get_feature_names_out(cat_cols), index=X_val.index)
X_test_cat = pd.DataFrame(encoder.transform(X_test[cat_cols]), columns=encoder.get_feature_names_out(cat_cols), index=X_test.index)

# Combine scaled numerical and encoded categorical features
X_train_processed = pd.concat([X_train_num, X_train_cat], axis=1)
X_val_processed = pd.concat([X_val_num, X_val_cat], axis=1)
X_test_processed = pd.concat([X_test_num, X_test_cat], axis=1)

In [14]:
# SMOTE is ONLY applied to the training set to prevent data leakage and ensure real-world testing.
smote = SMOTE(random_state=42)

print("Before SMOTE, y_train class distribution:")
print(y_train.value_counts())

X_train_smote, y_train_smote = smote.fit_resample(X_train_processed, y_train)

print("\nAfter SMOTE, y_train class distribution:")
print(y_train_smote.value_counts())

Before SMOTE, y_train class distribution:
readmitted
0    45655
1     4405
Name: count, dtype: int64

After SMOTE, y_train class distribution:
readmitted
0    45655
1    45655
Name: count, dtype: int64


In [17]:
# Define save directory
processed_dir = '../data/processed/'
os.makedirs(processed_dir, exist_ok=True)

# Save Train sets
X_train_smote.to_csv(os.path.join(processed_dir, 'X_train.csv'), index=False)
y_train_smote.to_csv(os.path.join(processed_dir, 'y_train.csv'), index=False)

# Save Validation sets
X_val_processed.to_csv(os.path.join(processed_dir, 'X_val.csv'), index=False)
y_val.to_csv(os.path.join(processed_dir, 'y_val.csv'), index=False)

# Save Test sets
X_test_processed.to_csv(os.path.join(processed_dir, 'X_test.csv'), index=False)
y_test.to_csv(os.path.join(processed_dir, 'y_test.csv'), index=False)

print("Blueprint Gate 2 PASSED: Preprocessed data successfully saved without target leakage.")

Blueprint Gate 2 PASSED: Preprocessed data successfully saved without target leakage.
